In [1]:
# Cell 1: Install dependencies
!pip install stable-baselines3[extra] supersuit "pettingzoo[atari]" "autorom[accept-rom-license]" -q
!AutoROM --accept-license

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms
	/usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms

Existing ROMs will be overwritten.


In [2]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Cell 3: Train
import os
import numpy as np
from pettingzoo.atari import warlords_v3
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback
from supersuit.vector.markov_vector_wrapper import MarkovVectorEnv
from supersuit.vector.concat_vec_env import ConcatVecEnv
from supersuit.vector.sb3_vector_wrapper import SB3VecEnvWrapper
from gymnasium.vector.utils import concatenate, create_empty_array

# Fix for SuperSuit bug: concat_obs ignores black_death flag
def _concat_obs_fixed(self, obs_dict):
    obs_list = []
    for agent in self.par_env.possible_agents:
        if agent not in obs_dict:
            if self.black_death:
                obs_list.append(np.zeros(self.observation_space.shape, dtype=self.observation_space.dtype))
            else:
                raise AssertionError("environment has agent death. Set black_death=True")
        else:
            obs_list.append(obs_dict[agent])
    return concatenate(
        self.observation_space,
        obs_list,
        create_empty_array(self.observation_space, self.num_envs),
    )

MarkovVectorEnv.concat_obs = _concat_obs_fixed

# DQN only supports n_envs=1
def make_env():
    env = MarkovVectorEnv(warlords_v3.parallel_env(obs_type="ram"), black_death=True)
    env = ConcatVecEnv([lambda e=env: e])
    env = SB3VecEnvWrapper(env)
    env = VecMonitor(env)
    return env

env = make_env()

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=64,
    gamma=0.99,
    exploration_fraction=0.2,
    exploration_final_eps=0.05,
    train_freq=4,
    target_update_interval=1000,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=1,
)

os.makedirs("/content/drive/MyDrive/warlords_checkpoints", exist_ok=True)

checkpoint_cb = CheckpointCallback(
    save_freq=250_000,
    save_path="/content/drive/MyDrive/warlords_checkpoints/",
    name_prefix="dqn_warlords_ram",
)

model.learn(total_timesteps=2_000_000, callback=[checkpoint_cb], progress_bar=True)
print("Training done!")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/base_vec_env.py:78: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Streaming output truncated to the last 5000 lines.
|    value_loss           | 0.00147    |
----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.9e+03     |
|    ep_rew_mean          | -1          |
| time/                   |             |
|    fps                  | 1968        |
|    iterations           | 7           |
|    time_elapsed         | 29          |
|    total_timesteps      | 57344       |
| train/                  |             |
|    approx_kl            | 0.007934659 |
|    clip_fraction        | 0.0954      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.77       |
|    explained_variance   | 0.0394      |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0999      |
|    n_updates            | 48          |
|    policy_gradient_loss | -0.000505   |
|    value_loss           | 0.0959      |
---------------------------

Training done!


In [6]:
# Cell 4: Save — run this IMMEDIATELY after training, before session disconnects!
import shutil, os

model.save("ppo_warlords_final")
env.save("ppo_warlords_vecnorm.pkl")

shutil.copy("ppo_warlords_final.zip",   "/content/drive/MyDrive/ppo_warlords_final.zip")
shutil.copy("ppo_warlords_vecnorm.pkl", "/content/drive/MyDrive/ppo_warlords_vecnorm.pkl")

print("Model size:",  os.path.getsize("/content/drive/MyDrive/ppo_warlords_final.zip"), "bytes")
print("VecNorm size:", os.path.getsize("/content/drive/MyDrive/ppo_warlords_vecnorm.pkl"), "bytes")
print("Both files saved to Drive!")

Model size: 2435254 bytes
VecNorm size: 3940 bytes
Both files saved to Drive!


In [7]:
# Cell 5: Sanity check — run before submitting to tournament
from stable_baselines3 import PPO
import numpy as np

model = PPO.load("ppo_warlords_final", device="cpu")
print("Timesteps trained:", model.num_timesteps)

actions = [
    int(model.predict(
        np.random.randint(0, 255, 128, dtype=np.uint8).astype(np.float32),
        deterministic=True
    )[0])
    for _ in range(30)
]

print("Actions:", actions)
print("Unique actions:", set(actions))

if len(set(actions)) == 1:
    print("WARNING: policy collapsed — try ent_coef=0.05 in Cell 3 and retrain")
else:
    print("OK: policy looks healthy, ready to submit!")

Timesteps trained: 2007040
Actions: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
Unique actions: {3}
